# Fix Parcor — Batch (Post-processor)

Post-processor for every `*_Parcor.csv` produced by the pipeline.

**Problems observed** on dictionary #4 (Jambi), confirmed across sample inspection:

1. **Unresolved `--` placeholders** — KBBI convention is to replace `--` with
   the current headword. 43.7% of source sentences contain unresolved `--`.
2. **Unresolved single `-` placeholders** — variant form the splitter missed.
   26.5% of source sentences contain this pattern.
3. **Cross-entry contamination** — multiple dictionary entries glued into a
   single row, with `kalimat_asal` and `kalimat_tujuan` potentially holding
   examples from *different* entries. 7% of rows have contamination indicators.
4. **Wrong-language content** — placeholder expansion sometimes put Indonesian
   words into the Jambi column.
5. **Token loss / truncation** — the splitter occasionally dropped tokens.

**Scope of this post-processor.**
- ✅ Expand `--` and `-` placeholders by tracing rows back to the preprocessing
     file and recovering the headword.
- ✅ Detect and flag contamination indicators.
- ✅ Detect and flag truncation / structural problems.
- ❌ Does NOT re-split contaminated rows (that requires upstream fixes).
- ❌ Does NOT fix wrong-language content (that requires upstream fixes).

Rows with flags are kept in the output but marked so downstream code can
filter them. A separate `_upstream_issues.csv` summarises what should be
fixed at earlier pipeline stages.

**Outputs**
- `../Ekstraksi/11. Parallel Corpus - Fixed/<id>_Parcor.csv` — cleaned, pipeline-compatible
- `../Ekstraksi/11. Parallel Corpus - Fixed/<id>_Parcor_audit.csv` — adds flag columns
- `../Ekstraksi/11. Parallel Corpus - Fixed/_summary.csv` — per-dict statistics
- `../Ekstraksi/11. Parallel Corpus - Fixed/_upstream_issues.csv` — bugs requiring
  pipeline-stage fixes, with example rows for each

## 1. Imports & paths

In [9]:
import re
from pathlib import Path
from typing import Optional

import pandas as pd

from _common import parse_dict_id, load_direction_lookup, direction_for

SRC_DIR = Path("../Ekstraksi/11. Parallel Corpus")
# Preprocessing output (where `contoh_kalimat` + `main_entry` live)
PREP_DIR = Path("../Ekstraksi/6. Pemecahan Definisi Lema")
# New output directory — does NOT overwrite "11. Parallel Corpus"
DST_DIR = Path("../Ekstraksi/11. Parallel Corpus - Fixed")

assert SRC_DIR.exists(), f"Source dir not found: {SRC_DIR.resolve()}"
DST_DIR.mkdir(parents=True, exist_ok=True)

print(f"Reading parcor from: {SRC_DIR.resolve()}")
print(f"Reading prep from:   {PREP_DIR.resolve()}")
print(f"Writing to:          {DST_DIR.resolve()}")

Reading parcor from: C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\11. Parallel Corpus
Reading prep from:   C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\6. Pemecahan Definisi Lema
Writing to:          C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\11. Parallel Corpus - Fixed


## 2. Helper — build a lemma index from the preprocessing file

For each dictionary we need a way to look up the headword (lemma) that a
given parcor sentence belongs to. The preprocessing file has this info
implicitly: `contoh_kalimat` contains the example sentences, and
`definisi_lema` has the headword.

Strategy: for each parcor row, find the prep row whose `contoh_kalimat`
*contains* the parcor sentence as a substring (using first 20 clean chars
as key). On dict 4 this matches ~99% of parcor rows.

In [10]:
def extract_lemma(definisi):
    """
    Extract the lemma (headword) from a `definisi_lema` string.
    Skip leading single-letter tokens (index markers like 'A', 'B')
    and digits (homonym numbering), returning the first real token.
    """
    if not isinstance(definisi, str):
        return ""
    for tok in definisi.strip().split():
        # Skip single letters (index markers) and bare digits (homonym markers)
        if len(tok) <= 1 or tok.isdigit():
            continue
        return tok.replace(".", "").lower()
    return ""


def clean_for_matching(s: str) -> str:
    """Normalise for substring matching — strip punctuation, lowercase."""
    return re.sub(r"[^\w\s]", "", str(s).lower()).strip()


def build_lemma_index(prep: pd.DataFrame):
    """
    Returns a list of (cleaned_contoh_kalimat, lemma, main_lemma) tuples.
    `main_lemma` is the most recent main_entry lemma (propagated downward
    so that sub-entries/derivations inherit their parent lemma).
    """
    prep = prep.copy()
    prep["lemma"] = prep["definisi_lema"].apply(extract_lemma)

    current_main = None
    main_lemmas = []
    for _, r in prep.iterrows():
        if r.get("main_entry") == 1:
            current_main = r["lemma"]
        main_lemmas.append(current_main if current_main else r["lemma"])
    prep["main_lemma"] = main_lemmas

    entries = []
    for _, r in prep.iterrows():
        if isinstance(r["contoh_kalimat"], str) and r["contoh_kalimat"].strip():
            entries.append((
                clean_for_matching(r["contoh_kalimat"]),
                r["lemma"],
                r["main_lemma"],
            ))
    return entries


def find_lemma_for_sentence(sentence: str, prep_entries, min_key_len=5):
    """Return (lemma, main_lemma) for this parcor sentence, or (None, None)."""
    key = clean_for_matching(sentence)[:20]
    if len(key) < min_key_len:
        return None, None
    for c_ex, lemma, main_lemma in prep_entries:
        if key in c_ex:
            return lemma, main_lemma
    return None, None

## 3. Placeholder expansion

KBBI uses `--` as the placeholder for the current headword; some dictionaries
also use a single `-`. We only expand placeholders when:
- we successfully identified the lemma for this row; AND
- the placeholder appears as a whole token (not mid-word).

We track how many placeholders were expanded vs. left unresolved.

In [11]:
# Match -- or - as standalone tokens (bounded by whitespace or sentence edges)
PLACEHOLDER_RE = re.compile(r"(?:^|(?<=\s))(--|-)(?=\s|$|[.,;:!?])")


def expand_placeholders(sentence: str, lemma: str) -> tuple[str, int, int]:
    """
    Replace standalone '--' or '-' with the given lemma.
    Returns (new_sentence, n_expanded, n_remaining).
    """
    if not isinstance(sentence, str) or not lemma:
        n_remaining = 0
        if isinstance(sentence, str):
            n_remaining = len(PLACEHOLDER_RE.findall(sentence))
        return sentence, 0, n_remaining

    n_expanded = 0
    def repl(m):
        nonlocal n_expanded
        n_expanded += 1
        return lemma

    new = PLACEHOLDER_RE.sub(repl, sentence)
    n_remaining = len(PLACEHOLDER_RE.findall(new))
    return new, n_expanded, n_remaining

## 4. Flags for contamination and structural problems

Rows get boolean flags (not dropped — downstream code can filter).
Flag names all start with `flag_` for clarity.

| Flag | Meaning |
|---|---|
| `flag_has_unresolved_placeholder` | Row still has `--` or `-` after expansion |
| `flag_has_syllable_pattern` | Indonesian side contains `x.yyy` dotted lemma pattern (next-entry bleed-in) |
| `flag_has_colon` | Indonesian side contains `:` (lemma-definition delimiter from next entry) |
| `flag_has_semicolon` | Indonesian side contains `;` (entry separator) |
| `flag_has_embedded_pos` | Indonesian side contains an isolated POS token mid-sentence |
| `flag_very_short_asal` | `kalimat_asal` has ≤2 tokens (likely truncated) |
| `flag_very_short_tujuan` | `kalimat_tujuan` has ≤2 tokens (likely truncated) |
| `flag_length_ratio_suspect` | token-ratio outside [0.4, 2.5] |
| `flag_lemma_not_found` | couldn't match row back to a lemma |

A row is `flag_any_contamination` if any of the contamination flags are set.
A row is `flag_clean` if no flags are set at all.

In [12]:
# Pre-compiled patterns
SYLLABLE_RE = re.compile(r"\b[a-z]\.[a-z]+")
POS_TOKENS = {"v", "a", "n", "pron", "adv", "num", "p", "vt", "vi"}
POS_TOKEN_RE = re.compile(
    r"(?<=\s)(" + "|".join(POS_TOKENS) + r")(?=\s)"
)


def compute_flags(asal: str, tujuan: str, lemma_found: bool) -> dict:
    a = str(asal) if isinstance(asal, str) else ""
    t = str(tujuan) if isinstance(tujuan, str) else ""

    n_tokens_a = len(a.split())
    n_tokens_t = len(t.split())

    flags = {
        "flag_has_unresolved_placeholder": bool(PLACEHOLDER_RE.search(a) or PLACEHOLDER_RE.search(t)),
        "flag_has_syllable_pattern": bool(SYLLABLE_RE.search(a)),
        "flag_has_colon": ":" in a,
        "flag_has_semicolon": ";" in a,
        "flag_has_embedded_pos": bool(POS_TOKEN_RE.search(a)),
        "flag_very_short_asal": n_tokens_a <= 2,
        "flag_very_short_tujuan": n_tokens_t <= 2,
        "flag_lemma_not_found": not lemma_found,
    }

    if n_tokens_a > 0 and n_tokens_t > 0:
        ratio = n_tokens_t / n_tokens_a
        flags["flag_length_ratio_suspect"] = ratio < 0.4 or ratio > 2.5
    else:
        flags["flag_length_ratio_suspect"] = True

    contamination_flags = [
        "flag_has_syllable_pattern", "flag_has_colon",
        "flag_has_semicolon", "flag_has_embedded_pos",
    ]
    flags["flag_any_contamination"] = any(flags[k] for k in contamination_flags)
    flags["flag_clean"] = not any(flags.values())

    return flags

## 5. Process one dictionary

In [13]:
def process_one(parcor_path: Path, direction_lookup: dict) -> Optional[dict]:
    dict_id = parse_dict_id(parcor_path.name)
    if dict_id is None:
        return None

    # Direction is not actually needed for the fix, but we report it.
    direction = direction_for(dict_id, direction_lookup)
    direction_known = direction is not None
    if not direction_known:
        direction = 1

    parcor = pd.read_csv(parcor_path)
    if "kalimat_asal" not in parcor.columns or "kalimat_tujuan" not in parcor.columns:
        return None

    # Load prep for this dictionary to build lemma index
    prep_candidates = list(PREP_DIR.glob(f"{dict_id}_*.csv"))
    if prep_candidates:
        prep = pd.read_csv(prep_candidates[0])
        prep_entries = build_lemma_index(prep)
    else:
        prep_entries = []
        print(f"  ⚠ No prep file for dict {dict_id}, placeholder expansion disabled")

    # Process each row
    records = []
    total_expanded_asal = 0
    total_expanded_tujuan = 0
    total_unresolved = 0
    lemma_matched = 0

    for _, row in parcor.iterrows():
        asal = row["kalimat_asal"] if isinstance(row["kalimat_asal"], str) else ""
        tujuan = row["kalimat_tujuan"] if isinstance(row["kalimat_tujuan"], str) else ""

        # Look up lemma
        lemma, main_lemma = find_lemma_for_sentence(asal, prep_entries)
        lemma_found = lemma is not None
        if lemma_found:
            lemma_matched += 1

        # Choose expansion target: main_lemma is usually the right one
        # (sub-entries expand to their parent lemma)
        expansion_lemma = main_lemma if main_lemma else lemma

        # Expand placeholders
        asal_new, exp_a, rem_a = expand_placeholders(asal, expansion_lemma or "")
        tujuan_new, exp_t, rem_t = expand_placeholders(tujuan, expansion_lemma or "")
        total_expanded_asal += exp_a
        total_expanded_tujuan += exp_t
        total_unresolved += rem_a + rem_t

        # Compute flags on the EXPANDED sentences (so we don't flag
        # successfully-expanded placeholders)
        flags = compute_flags(asal_new, tujuan_new, lemma_found)

        rec = {
            "kalimat_asal_original": asal,
            "kalimat_tujuan_original": tujuan,
            "kalimat_asal": asal_new,
            "kalimat_tujuan": tujuan_new,
            "lemma": lemma or "",
            "main_lemma": main_lemma or "",
            "placeholders_expanded_asal": exp_a,
            "placeholders_expanded_tujuan": exp_t,
            "placeholders_unresolved": rem_a + rem_t,
            **flags,
        }
        records.append(rec)

    audit_df = pd.DataFrame(records)

    # Pipeline-compatible output: just kalimat_asal, kalimat_tujuan
    compat_df = audit_df[["kalimat_asal", "kalimat_tujuan"]].copy()
    compat_df.to_csv(DST_DIR / f"{dict_id}_Parcor.csv", index=False)
    audit_df.to_csv(DST_DIR / f"{dict_id}_Parcor_audit.csv", index=False)

    flag_cols = [c for c in audit_df.columns if c.startswith("flag_")]
    summary = {
        "dict_id": dict_id,
        "direction": direction,
        "direction_known": direction_known,
        "rows": len(audit_df),
        "lemma_match_rate": round(lemma_matched / max(1, len(audit_df)), 3),
        "placeholders_expanded_asal": total_expanded_asal,
        "placeholders_expanded_tujuan": total_expanded_tujuan,
        "placeholders_unresolved": total_unresolved,
        "clean_rows": int(audit_df["flag_clean"].sum()),
        "contaminated_rows": int(audit_df["flag_any_contamination"].sum()),
    }
    # Per-flag counts
    for fc in flag_cols:
        summary[fc] = int(audit_df[fc].sum())

    return summary

## 6. Run on all dictionaries

In [14]:
direction_lookup = load_direction_lookup()

all_csvs = sorted(
    p for p in SRC_DIR.glob("*_Parcor.csv") if not p.name.startswith(".")
)
print(f"Found {len(all_csvs)} Parcor CSVs")

summaries = []
for i, csv in enumerate(all_csvs, 1):
    print(f"  [{i}/{len(all_csvs)}] {csv.name}")
    summary = process_one(csv, direction_lookup)
    if summary is not None:
        summaries.append(summary)

summary_df = pd.DataFrame(summaries).sort_values(
    "dict_id", key=lambda s: s.astype(int)
)
summary_df.to_csv(DST_DIR / "_summary.csv", index=False)
print(f"\nWrote summary to {DST_DIR / '_summary.csv'}")
summary_df

Found 68 Parcor CSVs
  [1/68] 10_Parcor.csv
  [2/68] 11_Parcor.csv
  [3/68] 12_Parcor.csv
  [4/68] 13_Parcor.csv
  [5/68] 14_Parcor.csv
  [6/68] 15_Parcor.csv
  [7/68] 16_Parcor.csv
  [8/68] 17_Parcor.csv
  [9/68] 18_Parcor.csv
  [10/68] 19_Parcor.csv
  [11/68] 1_Parcor.csv
  [12/68] 20_Parcor.csv
  [13/68] 21_Parcor.csv
  [14/68] 22_Parcor.csv
  [15/68] 23_Parcor.csv
  [16/68] 24_Parcor.csv
  [17/68] 25_Parcor.csv
  [18/68] 26_Parcor.csv
  [19/68] 27_Parcor.csv
  [20/68] 28_Parcor.csv
  [21/68] 29_Parcor.csv
  [22/68] 2_Parcor.csv
  [23/68] 31_Parcor.csv
  [24/68] 32_Parcor.csv
  [25/68] 33_Parcor.csv
  [26/68] 34_Parcor.csv
  [27/68] 35_Parcor.csv
  [28/68] 36_Parcor.csv
  [29/68] 37_Parcor.csv
  [30/68] 38_Parcor.csv
  [31/68] 3_Parcor.csv
  [32/68] 41_Parcor.csv
  [33/68] 42_Parcor.csv
  [34/68] 43_Parcor.csv
  [35/68] 44_Parcor.csv
  [36/68] 45_Parcor.csv
  [37/68] 46_Parcor.csv
  [38/68] 48_Parcor.csv
  [39/68] 4_Parcor.csv
  [40/68] 50_Parcor.csv
  [41/68] 51_Parcor.csv
  [42/68

,dict_id,direction,direction_known,rows,lemma_match_rate,placeholders_expanded_asal,placeholders_expanded_tujuan,placeholders_unresolved,clean_rows,contaminated_rows,...,flag_has_syllable_pattern,flag_has_colon,flag_has_semicolon,flag_has_embedded_pos,flag_very_short_asal,flag_very_short_tujuan,flag_lemma_not_found,flag_length_ratio_suspect,flag_any_contamination,flag_clean
10,1,0,True,2914,0.963,31,28,8,1109,886,...,24,576,725,0,869,563,107,228,886,1109
21,2,0,True,2047,0.872,1430,1028,42,464,808,...,4,68,782,4,665,541,261,415,808,464
30,3,0,True,125,0.880,1574,1687,184,11,81,...,12,80,31,29,32,25,15,17,81,11
38,4,1,True,2658,0.982,1797,378,12,2090,219,...,23,146,76,144,292,203,49,86,219,2090
47,5,1,True,697,0.795,1,2,0,109,172,...,11,8,132,48,464,406,143,47,172,109
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,85,0,True,532,0.964,368,352,37,354,123,...,2,109,93,49,43,32,19,23,123,354
62,87,1,True,1009,0.886,2696,2424,83,196,407,...,27,231,391,298,386,314,115,122,407,196
63,89,0,True,935,0.981,1,0,0,861,2,...,0,0,2,0,57,53,18,6,2,861
65,91,0,True,4762,0.988,3,0,0,4016,52,...,0,12,42,7,467,521,57,143,52,4016


## 7. Upstream issues report

These are the flag categories that require **upstream** fixes in earlier
notebooks — this post-processor can only detect them. Use this report to
prioritise which extraction/preprocessing bugs to tackle next.

In [15]:
upstream_issues = []

# For each flag, aggregate: count across dicts + a few example rows
flag_descriptions = {
    "flag_has_syllable_pattern": (
        "Indonesian side contains dotted syllable pattern (e.g. 'a.ba.di') — "
        "next-entry content bled in during preprocessing."
    ),
    "flag_has_colon": (
        "Indonesian side contains ':' — a lemma-definition delimiter suggests "
        "the next entry's headword was not properly separated."
    ),
    "flag_has_semicolon": (
        "Indonesian side contains ';' — this separator between entries was "
        "not used as a split point upstream."
    ),
    "flag_has_embedded_pos": (
        "Indonesian side has an isolated POS token mid-sentence — another "
        "next-entry bleed-in symptom."
    ),
    "flag_very_short_asal": (
        "kalimat_asal has <=2 tokens — likely truncated during splitting."
    ),
    "flag_length_ratio_suspect": (
        "Indonesian / Jambi token ratio outside [0.4, 2.5] — likely "
        "misaligned pair or truncated."
    ),
}

for flag, description in flag_descriptions.items():
    if flag not in summary_df.columns:
        continue
    total = int(summary_df[flag].sum())
    affected_dicts = int((summary_df[flag] > 0).sum())

    # Find an example
    example = ""
    for csv in all_csvs[:5]:  # check first 5 dicts for a concrete example
        dict_id = parse_dict_id(csv.name)
        audit_path = DST_DIR / f"{dict_id}_Parcor_audit.csv"
        if audit_path.exists():
            aud = pd.read_csv(audit_path)
            hit = aud[aud[flag].astype(bool)] if flag in aud.columns else pd.DataFrame()
            if len(hit) > 0:
                ex = hit.iloc[0]
                example = f"[dict {dict_id}] asal={ex['kalimat_asal_original'][:80]}"
                break

    upstream_issues.append({
        "flag": flag,
        "total_rows_affected": total,
        "dictionaries_affected": affected_dicts,
        "description": description,
        "example": example,
    })

upstream_df = pd.DataFrame(upstream_issues).sort_values(
    "total_rows_affected", ascending=False
)
upstream_df.to_csv(DST_DIR / "_upstream_issues.csv", index=False)

print("Upstream issues ranked by volume (these require fixes in earlier notebooks):\n")
for _, r in upstream_df.iterrows():
    print(f"  {r['flag']:<35}  {r['total_rows_affected']:>6} rows  "
          f"({r['dictionaries_affected']} dicts)")
    print(f"    {r['description']}")
    if r['example']:
        print(f"    e.g. {r['example']}")
    print()

Upstream issues ranked by volume (these require fixes in earlier notebooks):

  flag_very_short_asal                  34036 rows  (68 dicts)
    kalimat_asal has <=2 tokens — likely truncated during splitting.
    e.g. [dict 10] asal=gergaji

  flag_has_semicolon                    17919 rows  (68 dicts)
    Indonesian side contains ';' — this separator between entries was not used as a split point upstream.
    e.g. [dict 10] asal=ia ~ benih illl sebelum menanamnya, i se ngehambur luwonun wini sio sabalupm ndu

  flag_has_colon                        12464 rows  (67 dicts)
    Indonesian side contains ':' — a lemma-definition delimiter suggests the next entry's headword was not properly separated.
    e.g. [dict 10] asal=penyakilnya se makin gawat, keadaatn rototnne mangkitn ries ad at n ada!: menurl

  flag_length_ratio_suspect             12226 rows  (68 dicts)
    Indonesian / Jambi token ratio outside [0.4, 2.5] — likely misaligned pair or truncated.
    e.g. [dict 10] asal=Tuhan 

## 8. Totals

In [16]:
print(f"Dictionaries processed:           {len(summary_df)}")
print(f"Total rows:                       {summary_df['rows'].sum():,}")
print(f"Lemma match rate (avg):           {summary_df['lemma_match_rate'].mean():.1%}")
print(f"Placeholders expanded (asal):     {summary_df['placeholders_expanded_asal'].sum():,}")
print(f"Placeholders expanded (tujuan):   {summary_df['placeholders_expanded_tujuan'].sum():,}")
print(f"Placeholders remaining unresolved: {summary_df['placeholders_unresolved'].sum():,}")
print(f"\nClean rows:                       {summary_df['clean_rows'].sum():,} "
      f"({summary_df['clean_rows'].sum()/summary_df['rows'].sum():.1%})")
print(f"Contaminated rows:                {summary_df['contaminated_rows'].sum():,} "
      f"({summary_df['contaminated_rows'].sum()/summary_df['rows'].sum():.1%})")
print(f"\nOutputs written to: {DST_DIR.resolve()}")

Dictionaries processed:           68
Total rows:                       116,398
Lemma match rate (avg):           87.2%
Placeholders expanded (asal):     66,566
Placeholders expanded (tujuan):   42,223
Placeholders remaining unresolved: 1,858

Clean rows:                       53,027 (45.6%)
Contaminated rows:                24,235 (20.8%)

Outputs written to: C:\Users\Legion\OneDrive\Documents\UNI\TA\tugas-akhir-data-mining\TAEkstraksiKamus\Ekstraksi\11. Parallel Corpus - Fixed
